# NB2 — Dữ liệu sở thích (preference) tiếng Việt

**Bộ dữ liệu mặc định:** `sailor2/sea-ultrafeedback-onpolicy`, lọc `language == "Vietnamese"`
(khoảng 4.1k cặp). Lab cũ huấn luyện DPO trên UltraFeedback tiếng Anh trong khi SFT và
đánh giá đều bằng tiếng Việt, nên khó đọc hiệu ứng của DPO.

> **Mục tiêu:** đưa dữ liệu về dạng hội thoại của TRL
> (`prompt`/`chosen`/`rejected` là list message), lọc cặp vượt `MAX_LEN`,
> chia **tập huấn luyện/eval không trùng câu hỏi**, đo thiên vị độ dài, lưu Parquet.
>
> **Giấy phép:** bộ dữ liệu không ghi giấy phép. Nguồn gốc là UltraFeedback (câu hỏi) và
> phản hồi do mô hình sinh rồi được chấm, nên chỉ dùng cho học tập và nghiên cứu.
> Đặt `PREF_DATASET=argilla/ultrafeedback-binarized-preferences-cleaned PREF_LANGUAGE=`
> để chạy lại bản tiếng Anh làm đối chứng.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from transformers import AutoTokenizer

from lab22 import config as C
from lab22 import data as D

C.ensure_dirs()
print(C.summary())

# Only the tokenizer is needed here: chat template + length budget. No GPU.
tokenizer = AutoTokenizer.from_pretrained(C.BASE_MODEL)

/opt/day22-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


COMPUTE_TIER=T4 base=Qwen/Qwen3-0.6B max_len=768 seed=42
SFT saillab/alpaca-vietnamese-cleaned[:1000]  PREF sailor2/sea-ultrafeedback-onpolicy (Vietnamese) train=800 eval=100
DPO beta=0.1 lr=5e-06 epochs=1.0 loss=['sigmoid']


## 1. Nạp, lọc, chia huấn luyện/eval theo câu hỏi

In [2]:
train_ds, eval_ds = D.load_preference_pairs(
    C.PREF_DATASET,
    tokenizer,
    max_len=C.MAX_LEN,
    n_train=C.PREF_TRAIN,
    n_eval=C.PREF_EVAL,
    language=C.PREF_LANGUAGE or None,
    seed=C.SEED,
    template_kwargs=C.CHAT_TEMPLATE_KWARGS,
)
D.assert_disjoint(list(train_ds), list(eval_ds))
print(f"train={len(train_ds)}  eval={len(eval_ds)}  (no prompt overlap)")
for index, row in enumerate(train_ds.select(range(min(3, len(train_ds))))):
    print(f"\nExample {index + 1}\nPrompt: {row['prompt'][0]['content']}\n"
          f"Chosen: {row['chosen'][0]['content']}\nRejected: {row['rejected'][0]['content']}")

Generating train split:   0%|          | 0/38327 [00:00<?, ? examples/s]

Generating train split:   3%|▎         | 1000/38327 [00:00<00:12, 2920.16 examples/s]

Generating train split:   8%|▊         | 3000/38327 [00:00<00:04, 7885.99 examples/s]

Generating train split:  13%|█▎        | 5000/38327 [00:00<00:03, 10877.96 examples/s]

Generating train split:  18%|█▊        | 7000/38327 [00:00<00:02, 12634.16 examples/s]

...

Filter:  94%|█████████▍| 36000/38327 [00:12<00:00, 3217.56 examples/s]

Filter:  97%|█████████▋| 37000/38327 [00:12<00:00, 3789.67 examples/s]

Filter:  99%|█████████▉| 38000/38327 [00:12<00:00, 4353.61 examples/s]

Filter: 100%|██████████| 38327/38327 [00:12<00:00, 2952.26 examples/s]

train=800  eval=100  (no prompt overlap)

Example 1
Prompt: Tạo 10 yêu cầu thay đổi.

--- ví dụ ---
Trước: Một cô gái cưỡi ngựa
Yêu cầu: Biến cô ấy thành cưỡi kỳ lân
Sau: Một cô gái cưỡi kỳ lân
--- kết thúc ví dụ ---
Chosen: **10 Yêu cầu Thay đổi:**

1. **Trước:** Một thị trấn xưa với xe ngựa
**Yêu cầu:** Biến phương tiện giao thông thành xe bay
**Sau:** Một thị trấn cổ với xe bay lơ lửng

2. **Trước:** Bữa trà chiều truyền thống trong một ngôi nhà nông thôn
**Yêu cầu:** Thêm các yếu tố tương lai (robot phục vụ, đồ nội thất tự thích nghi)
**Sau:** Bữa trà "cyber-chic" với công nghệ tiên tiến

3. **Trước:** Một nhà thám hiểm trong rừng
**Yêu cầu:** Trao cho anh ta khả năng giao tiếp với thực vật
**Sau:** Nhà thám hiểm sinh thái giao tiếp với rừng rậm

4. **Trước:** Cảnh bãi biển mùa hè cổ điển
**Yêu cầu:** Thêm các cấu trúc bãi biển dưới nước đa dạng sinh học
**Sau:** Bãi biển đầy năng lượng tự duy trì với rạn san hô tinh tế

5. **Trước:** Thư viện sách truyền thống
**Yêu cầu:** Chuyển 

## 2. Thiên vị độ dài

Nếu phần lớn `chosen` dài hơn `rejected`, DPO có thể học "viết dài hơn" thay vì
"trả lời tốt hơn". Ghi con số này vào REFLECTION và so với độ dài đầu ra ở NB4.

In [3]:
def n_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


stats = D.length_stats(list(train_ds), count=n_tokens)
print(f"chosen median {stats['chosen_median']:.0f} tok · rejected median {stats['rejected_median']:.0f} tok")
print(f"chosen longer in {stats['chosen_longer_frac']:.1%} of pairs")

chosen median 94 tok · rejected median 86 tok
chosen longer in 65.9% of pairs


In [4]:
import matplotlib.pyplot as plt
import numpy as np

chosen = np.array([n_tokens(r["chosen"][0]["content"]) for r in train_ds])
rejected = np.array([n_tokens(r["rejected"][0]["content"]) for r in train_ds])
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, C.MAX_LEN, 40)
ax.hist(chosen, bins=bins, alpha=0.6, label="chosen", color="#2e548a")
ax.hist(rejected, bins=bins, alpha=0.6, label="rejected", color="#c83538")
ax.set_xlabel("response tokens")
ax.set_title(f"Length: chosen longer in {stats['chosen_longer_frac']:.0%} of pairs")
ax.legend()
fig.savefig(C.SCREENSHOTS / "02b-pref-length.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Lưu

In [5]:
import json

train_ds.to_parquet(str(C.PREF_DIR / "train.parquet"))
eval_ds.to_parquet(str(C.PREF_DIR / "eval.parquet"))
(C.PREF_DIR / "stats.json").write_text(
    json.dumps({"dataset": C.PREF_DATASET, "language": C.PREF_LANGUAGE,
                "base_model": C.BASE_MODEL, "max_length": C.MAX_LEN, "seed": C.SEED,
                "train_pairs": len(train_ds), "eval_pairs": len(eval_ds), **stats}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved {len(train_ds)} train / {len(eval_ds)} eval pairs → {C.PREF_DIR}")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 36.66ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 248.12ba/s]

Saved 800 train / 100 eval pairs → /lab/data/pref


## 4. Ghi chú vibe-coding

Mở 5 cặp ngẫu nhiên và tự chấm: bạn có đồng ý với nhãn `chosen` không? Khoảng 1%
câu `chosen` trong bộ này lẫn tiếng Anh hoặc mất dấu, một số câu hỏi là bài code.
Nếu bạn lọc thêm (ví dụ bỏ cặp lệch độ dài > 2×), ghi lại số cặp còn lại và lý do
vào REFLECTION. `BONUS-CHALLENGE.md` #1 gợi ý tự xây dữ liệu sở thích (preference) tiếng Việt gốc.

**Tiếp theo:** NB3 — huấn luyện DPO.